# Yapay Zeka ve Üretken Modeller Atölyesi - 6
## Doğal Dil İşleme (NLP) ve Metin Analizi

**BTK Akademi** | Dr.Öğr.Üyesi Furkan Göz

---

Bu notebook, BTK Akademi Yapay Zeka ve Üretken Modeller Atölyesi'nin 6. haftasında işlenen konuları kapsamaktadır.

### İçindekiler
1. **Doğal Dil İşleme (NLP) Nedir?**
2. **NLP Görevleri** - Metin ve Kelime Düzeyinde İşlemler
3. **Uygulama 1: Varlık İsmi Tanıma (NER)**
4. **Uygulama 2: Otomatik Metin Özetleme**
5. **Duygu Analizi**
6. **Düzenli İfadeler (Regex)**
7. **Metin Ön İşleme (Preprocessing)**
8. **Metin Temsilleri** - BoW, TF-IDF, N-Gram
9. **Kelime Gömme (Word Embeddings)** - Word2Vec, GloVe, FastText
10. **Sıralı Veriler ve RNN Mimarileri** - RNN, LSTM, GRU
11. **LSTM ile Metin Sınıflandırma Uygulaması**
12. **Model Değerlendirme Metrikleri**

---
## Gerekli Kütüphanelerin Kurulumu

Aşağıdaki hücreyi çalıştırarak gerekli kütüphaneleri yükleyin.

In [ ]:
# Gerekli kutuphanelerin kurulumu
# !pip install transformers torch nltk textblob scikit-learn tensorflow numpy matplotlib seaborn

---
## 1. Doğal Dil İşleme (NLP) Nedir?

Doğal Dil İşleme (NLP), bilgisayarların insan dilini anlamasını ve işlemesini sağlayan yapay zeka alanıdır.

**Temel Özellikler:**
- Metin ve ses verileri üzerinde çalışarak düzensiz bilgiyi bilgisayarlar için anlamlı hale getirir.
- Uygulama alanları: sohbet robotları, makine çevirisi, duygu analizi
- Temel amaç: bilgisayarların insan iletişimini doğru şekilde yorumlayabilmesi

### NLP Ne Yapmayı Hedefler?
- Metinlerden bağlam ve anlam çıkarmak
- Uzun belgeleri otomatik olarak özetlemek
- Soru-cevap sistemleri ve arama motorları geliştirmek
- Ham yazıları bilgisayarların işleyebileceği formata dönüştürmek

---
## 2. NLP Görevleri

### Metin Düzeyinde İşlemler
| Görev | Açıklama |
|-------|----------|
| **Metin Sınıflandırma** | Gelen metnin kategorisini belirleme |
| **Makine Çevirisi** | Metinleri bir dilden diğerine otomatik çevirme |
| **Metin Özetleme** | Uzun yazılardan temel bilgileri çıkarma |

### Kelime Düzeyinde İşlemler
| Görev | Açıklama |
|-------|----------|
| **Varlık Tanıma (NER)** | Metin içindeki kişi, yer ve kurum isimlerini bulma |
| **Cümle Benzerliği** | İki farklı cümlenin aynı anlama gelip gelmediğini kontrol etme |

### Diğer NLP Görevleri
- **Soru-Cevap Sistemleri:** Verilen bir metni okuyup sorulan soruya doğrudan yanıt üretme
- **Sıfır Örnekle Sınıflandırma (Zero-Shot):** Modelin eğitim aşamasında görmediği yeni konuları tahmin etmesi
- **Diyalog Sistemleri:** İnsanlarla doğal şekilde konuşabilen sohbet robotları geliştirme

---
## 3. Uygulama 1: Varlık İsmi Tanıma (NER)

Metin içindeki özel isim, kurum ve tarih bilgilerini otomatik etiketleme işlemidir.

**Kategoriler:**
- **Kişi (PER):** Şahıs isimlerini tespit eder
- **Organizasyon (ORG):** Şirket, dernek ve kurum adlarını bulur
- **Konum (LOC):** Ülke, şehir ve coğrafi bölge bilgilerini işaretler

Model, metni okurken her bir kelimenin hangi kategoriye ait olduğunu istatistiksel olarak hesaplar.

### 3.1 Genel NER Pipeline (İngilizce)

In [ ]:
from transformers import pipeline

# NER pipeline olusturma - varsayilan model ile
ner_model = pipeline("ner", aggregation_strategy="simple")

text = "Aselsan sirketi Kocaeli merkezinde 2026'da yeni bir ofis aciyor."

results = ner_model(text)

for entity in results:
    print("Tespit Edilen Kelime:", entity['word'])
    print("Varlik Kategorisi:", entity['entity_group'])
    print("Guven Skoru:", round(entity['score'], 4))
    print("---")

### 3.2 Türkçe NER Uygulaması

Türkçe metinler için özel olarak eğitilmiş BERT tabanlı model kullanıyoruz.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Turkce NER modeli yukleme
model_id = "akdeniz27/bert-base-turkish-cased-ner"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(model_id)

ner_pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

text = "Ahmet Yilmaz yarin Kocaeli Üniversitesi'nde konusma yapacak."

results = ner_pipe(text)

for entity in results:
    print("Kelime:", entity['word'])
    print("Kategori:", entity['entity_group'])
    print("Guven Skoru:", round(entity['score'], 4))
    print("---")

### 3.3 Sağlık Sektöründe Varlık Tanıma

NER'in sağlık alanındaki kullanımları:
- Doktor notları ve hasta raporları üzerinden tıbbi terimlerin ayrıştırılması
- Hastalık adları, ilaç isimleri ve dozaj bilgilerinin otomatik yapılandırılması
- Hastane bilgi yönetim sistemleri için istatistiksel rapor üretimi
- İnsan kaynaklı veri giriş hatalarının minimize edilmesi

---
## 4. Uygulama 2: Otomatik Metin Özetleme

Uzun makale ve belgeleri ana fikri koruyarak kısa boyutlara indirme işlemidir.

**İki Temel Yaklaşım:**

| Yöntem | Açıklama |
|--------|----------|
| **Çıkarımsal (Extractive)** | Metin içindeki en önemli cümleler puanlanır ve orijinal haliyle kopyalanarak birleştirilir |
| **Soyutlayıcı (Abstractive)** | Model tüm metni anladıktan sonra orijinal belgede bulunmayan yeni kelimelerle insan gibi özet yazar |

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Turkce ozetleme modeli
model_adi = "mukayese/transformer-turkish-summarization"
tokenizer = AutoTokenizer.from_pretrained(model_adi)
model = AutoModelForSeq2SeqLM.from_pretrained(model_adi)

text = """Yapay zeka sistemleri veri setlerindeki desenleri analiz ederek ogrenir. 
Bu teknoloji saglik ve otomotiv gibi bircok farkli endustride kullanilmaktadir. 
Gelecekte otonom sistemlerin daha da yayginlasmasi beklenmektedir."""

inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(inputs.input_ids, max_length=40)

print("Uretilen Ozet:", tokenizer.decode(outputs[0], skip_special_tokens=True))

---
## 5. Duygu Analizi (Sentiment Analysis)

### Sistem İşleyişi
- Ağ, olumlu kelimelerin ağırlıkta olduğu dizilimlere **pozitif** etiketi atar
- Olumsuzluk içeren kelime dizilimleri **negatif** etiketi ile sınıflandırılır
- "Harika" → pozitif | "Berbat" → negatif
- Sistem belirsiz olan yorumları eşik değer denetimi ile manuel onaya gönderebilir

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Turkce duygu analizi modeli
model_adi = "savasy/bert-base-turkish-sentiment-cased"
tokenizer = AutoTokenizer.from_pretrained(model_adi)
model = AutoModelForSequenceClassification.from_pretrained(model_adi)

duygu_analizi = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

text = "Siparis verdigim urun cok hizli elime ulasti ve kalitesi harika."
sonuc = duygu_analizi(text)

print("Duygu Sinifi:", sonuc[0]['label'])
print("Olasilik Skoru:", round(sonuc[0]['score'], 4))

In [ ]:
# Birden fazla cumle ile test edelim
test_cumleleri = [
    "Bu urun gercekten harika, cok memnunum!",
    "Kargo cok gec geldi, urun de bozuk cikti.",
    "Fiyatina gore fena degil ama daha iyisi olabilirdi.",
    "Tekrar satin alirim, kalite mukemmel."
]

for cumle in test_cumleleri:
    sonuc = duygu_analizi(cumle)
    print(f"Metin: {cumle}")
    print(f"  -> Duygu: {sonuc[0]['label']} | Skor: {round(sonuc[0]['score'], 4)}")
    print()

---
## 6. Düzenli İfadeler (Regex)

Metin içinde belirli desenleri aramak ve bulmak için kullanılan bir kural setidir. NLP projelerinde veri temizleme aşamasında yardımcı araç olarak kullanılır.

### Temel Regex Karakterleri

| Karakter | İşlev |
|----------|-------|
| `.` | Herhangi bir tek karakteri temsil eder |
| `*` | Önceki karakterin sıfır veya daha fazla tekrarı |
| `+` | Önceki karakterin en az bir kez tekrarı |
| `?` | Önceki karakter isteğe bağlı |
| `[abc]` | Belirtilen karakterlerden herhangi biri |
| `\d` | Sadece rakamları bulur |
| `\w` | Harf, rakam ve alt çizgi |
| `\s` | Boşluk ve sekme karakterleri |
| `\S` | Boşluk dışındaki tüm karakterler |
| `{n}` | Tam n kez tekrar |
| `{n,}` | En az n kez tekrar |
| `^` | Satır başı |
| `$` | Satır sonu |

### Python Regex Fonksiyonları
- `re.search()` → İlk eşleşmeyi bulur
- `re.findall()` → Tüm eşleşmeleri liste halinde toplar
- `re.sub()` → Bulunan desenleri yeni metin ile değiştirir
- `re.split()` → Metni kurala göre parçalara böler

In [ ]:
import re

text = "Ulasim icin bilgi@firma.com veya 0555-444-3322 kullanin."

# E-posta adreslerini bulma
emails = re.findall(r"\S+@\S+\.\S+", text)
print("E-posta:", emails)

# Telefon numaralarini bulma
phones = re.findall(r"\d{4}-\d{3}-\d{4}", text)
print("Telefon:", phones)

In [ ]:
# Ek Regex ornekleri

# URL bulma
text2 = "Sitemiz https://www.ornek.com adresindedir. Bilgi icin http://bilgi.org"
urls = re.findall(r"https?://\S+", text2)
print("URL'ler:", urls)

# Ozel karakterleri temizleme
text3 = "Merhaba!!! Bu bir #test... @kullanici ile"
temiz = re.sub(r"[^a-zA-ZıİğĞüÜşŞöÖçÇ\s]", "", text3)
print("Temizlenmis:", temiz)

# Cumleyi bolme
text4 = "Python,Java,C++,Rust"
diller = re.split(r",", text4)
print("Diller:", diller)

---
## 7. Metin Ön İşleme (Preprocessing)

Ham metni makine öğrenmesi modellerinin anlayabileceği temiz bir formata dönüştürme sürecidir.

### Ön İşleme Adımları
1. **Temizleme:** Etiketleri, sembolleri ve gereksiz karakterleri silme
2. **Standartlaştırma:** Harfleri küçültme ve tarih/sayı formatlarını eşitleme
3. **Ayrıştırma (Tokenization):** Cümleyi kelimelere bölme
4. **Durdurma Kelimeleri (Stopwords):** Anlam taşımayan bağlaçları çıkarma
5. **Kök Bulma:** Kelimeleri eklerinden arındırma

### 7.1 Ayrıştırma (Tokenization)

Metni bilgisayarın işleyebileceği en küçük birimlere ayırma işlemidir.

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.tokenize import word_tokenize

text = "U.S.A. uzerindeki satislar 12.40 dolardir."

# NLTK Kutuphanesi ile ayristirma
print("NLTK:", word_tokenize(text))

In [ ]:
# Keras fonksiyonu ile ayristirma
from tensorflow.keras.preprocessing.text import text_to_word_sequence

print("Keras:", text_to_word_sequence(text))

**Not:** NLTK, kısaltmaları (U.S.A.) korurken Keras tüm noktalama işaretlerini kaldırır ve küçük harfe dönüştürür. Projenizin ihtiyacına göre uygun tokenizer seçilmelidir.

### 7.2 Durdurma Kelimeleri (Stopwords)

Cümle kurmak için gerekli olan ancak tek başına anlam taşımayan kelimelerdir. "ve", "ile", "veya" gibi bağlaçlar veri setinden çıkarılır.

**Dikkat:** Makine çevirisi gibi cümlenin tam yapısının gerektiği durumlarda stopwords çıkarılmamalıdır!

In [ ]:
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

text = "Bu cumle ornek bir test metnidir ve analiz edilecektir."
tokens = word_tokenize(text)

# Turkce durdurma kelimelerini yukleme
stop_words = set(stopwords.words('turkish'))
print(f"Turkce stopword sayisi: {len(stop_words)}")
print(f"Ornek stopwords: {list(stop_words)[:10]}")
print()

filtered_tokens = [w for w in tokens if w.lower() not in stop_words]

print("Orijinal:", tokens)
print("Temizlenmis Veri:", filtered_tokens)

### 7.3 Kök Bulma (Stemming ve Lemmatization)

Kelimeleri eklerinden temizleyerek kök formuna getirme işlemidir.

| Yöntem | Açıklama | Hız | Doğruluk |
|--------|----------|-----|----------|
| **Stemming** | Kelime sonundaki ekleri basit kurallarla keser | Hızlı | Bazen anlamsız sonuçlar |
| **Lemmatization** | Kelimenin sözlük anlamına bakarak doğru kökü bulur | Yavaş | Tamamen anlamlı |

In [ ]:
import nltk
nltk.download('wordnet', quiet=True)

from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Farkli kelimelerle karsilastirma
test_words = ["studies", "studying", "studied", "running", "better", "flies"]

print(f"{'Kelime':<15} {'Stemming':<15} {'Lemmatization':<15}")
print("-" * 45)
for word in test_words:
    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos="v")
    print(f"{word:<15} {stem:<15} {lemma:<15}")

---
## 8. Metin Temsilleri (Text Representations)

Bilgisayarlar metin formatındaki karakterleri doğrudan anlayamaz. Makine öğrenmesi modellerinin çalışabilmesi için kelimelerin sayılara dönüştürülmesi gerekir.

### 8.1 Kelime Çantası (Bag-of-Words) Modeli

Kelimelerin belge içindeki geçme sayısına dayanan en temel metin temsil yöntemidir.

**Özellikler:**
- Kelimelerin sırasını ve dil bilgisi kurallarını tamamen yok sayar
- Sadece hangi kelimenin kaç kez kullanıldığı bilgisini tutar
- Basit yapısına rağmen sınıflandırma görevlerinde hızlı sonuç verir

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "Sistem cok iyi calisiyor",
    "Sistem yavas ve kotu",
    "Harika bir sonuc elde edildi"
]

# Modeli kurma ve veriyi sayisal dizilere donusturme
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

print("Sozluk:", vectorizer.get_feature_names_out())
print("\nSayisal Agirliklar:")
print(X.toarray())

### 8.2 TF-IDF Modeli

TF-IDF, kelimenin belge içindeki ayırt edici gücünü ve önemini hesaplar.

**Formül:**
- **TF (Terim Frekansı):** Kelimenin tek bir belgedeki geçme sıklığı
- **IDF (Ters Doküman Frekansı):** Kelimenin tüm veri setindeki bulunma nadirliği
- **TF-IDF = TF × IDF**

Sık tekrarlanan ama bilgi taşımayan kelimelerin etkisini düşürürken, nadir kullanılan kelimelere yüksek ağırlık vererek modelin asıl konuya odaklanmasını sağlar.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

documents = [
    "Sistem cok iyi calisiyor",
    "Sistem yavas ve kotu",
    "Harika bir sonuc elde edildi"
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(documents)

print("Sozluk:", vectorizer.get_feature_names_out())
print("\nTF-IDF Agirliklari:")
# Daha okunakli format
tfidf_array = X.toarray()
for i, doc in enumerate(documents):
    print(f"\nBelge {i+1}: '{doc}'")
    for j, word in enumerate(vectorizer.get_feature_names_out()):
        if tfidf_array[i][j] > 0:
            print(f"  {word}: {tfidf_array[i][j]:.4f}")

### 8.3 N-Gram Modeli

N-gram, metin içindeki ardışık kelime gruplarını ifade eder.

| Tür | Açıklama | Örnek ("Bugün hava güzel") |
|-----|----------|---------------------------|
| **Unigram (1-gram)** | Tekil kelimeler | bugün, hava, güzel |
| **Bigram (2-gram)** | İkili kelime grupları | bugün hava, hava güzel |
| **Trigram (3-gram)** | Üçlü kelime dizilimleri | bugün hava güzel |

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "Bugun cok mutlu hissediyorum",
    "Asiri uzgun ve yorgunum"
]

# Unigram + Bigram (1-gram ve 2-gram)
vectorizer = CountVectorizer(ngram_range=(1, 2))
# her bir kelimeyi vektör olarak alıyor 
X = vectorizer.fit_transform(docs)

print("N-Gram Sozlugu:")
for ngram in vectorizer.get_feature_names_out():
    print(f"  - {ngram}")

### 8.4 Klasik Yöntemlerin Sınırları

- BoW ve TF-IDF kelimelerin sırasını ve anlamını dikkate almaz
- N-Gram kelime sırasını kullanır ancak kelimeler arası anlam bağını kuramaz
- Geleneksel yöntemler "kedi" ve "köpek" kelimelerinin hayvan olduğunu bilemez
- **Çözüm:** Derin anlamsal temsillere (Word Embeddings) ihtiyaç duyulur

---
## 9. Kelime Gömme (Word Embeddings)

Her kelimeyi çok boyutlu bir uzayda koordinat noktası olarak tanımlar. Anlamı benzeyen kelimeler sanal uzayda birbirine yakın konumlanır.

### Dağılımsal Semantik
Bir kelimenin anlamı, etrafındaki diğer kelimeler ile belirlenir. Benzer cümlelerde geçen farklı kelimeler benzer anlamlar taşır.

### 9.1 Word2Vec

Kelimenin anlamını komşu kelimelerin yardımıyla hesaplayan popüler bir modeldir.

**İki Yöntem:**
- **CBOW (Sürekli Kelime Çantası):** Bağlam kelimelerinden ortadaki kelimeyi tahmin eder → Küçük veri setlerinde hızlı
- **Skip-Gram:** Ortadaki kelimeyi kullanarak etraftaki kelimeleri tahmin eder → Nadir kelimelerde başarılı

**Vektör Matematiği Örnekleri:**
```
vector[Kral] - vector[Erkek] + vector[Kadın] ≈ vector[Kraliçe]
vector[Paris] - vector[Fransa] + vector[İtalya] ≈ vector[Roma]
vector[Ankara] - vector[Türkiye] + vector[Almanya] ≈ vector[Berlin]
```

### 9.2 GloVe Modeli

- Sadece yan yana duran kelimelere değil, tüm metin setindeki ilişkilere bakar
- Kelimelerin veri setinde birlikte geçme sayılarını tablo halinde tutar
- Küresel verileri kullanarak daha kapsamlı bir kelime haritası çıkarır

### 9.3 FastText Modeli

- Kelimeleri bir bütün olarak değil, daha küçük harf grupları (subword) halinde inceler
- Örnek: "kedi" → ke, ked, edi, di
- Eğitim verisinde hiç bulunmayan kelimeler için alt parçaları kullanarak anlam üretir (OOV handling)
- **Türkçe gibi sondan eklemeli (agglutinative) dillerde çok başarılıdır!**

---
## 10. Sıralı Veriler ve RNN Mimarileri

### Sıralı Veriler (Sequential Data)
Metin, ses ve zaman serileri, olayların dizilim sırası ile anlam kazandığı veri türleridir. Kelimelerin yer değiştirmesi genel anlamı tamamen değiştirir.

### 10.1 Tekrarlayan Sinir Ağları (RNN)

- Önceki adımlarda üretilen çıktıları yeni kelimelerle birleştirir
- Her adımda hem mevcut kelimeyi hem de geçmişten gelen gizli durum (hidden state) bilgisini işler
- Ağın kendi içinde ardışık bir hafıza oluşturmasını sağlar

**Problem: Vanishing Gradient (Hafıza Kaybı)**
- Uzun cümlelerde ilk kelimelerin bilgisi son kelimelere kadar taşınamaz
- Geri yayılım sürecinde ağırlık değerleri sürekli çarpılarak sıfıra yaklaşır
- Model cümlenin başındaki bağlamı unutur

### 10.2 LSTM (Uzun Kısa Süreli Hafıza)

Hafıza kaybı problemini çözmek amacıyla özel **kapı mekanizmaları** kullanır.

| Kapı | İşlev |
|------|-------|
| **Unutma Kapısı (Forget Gate)** | Önceki adımlardan gelen gereksiz bilgilerin yüzde kaçının silineceğini belirler |
| **Giriş Kapısı (Input Gate)** | Yeni gelen kelimenin hangi kısımlarının hafızaya ekleneceğini belirler |
| **Çıkış Kapısı (Output Gate)** | Güncellenmiş hafızayı kullanarak bir sonraki hücreye aktarılacak durumu üretir |

### 10.3 GRU (Geçitli Tekrarlayan Birim)

LSTM'in daha sade alternatifi — sadece 2 kapı: **Güncelleme** ve **Sıfırlama**

| Özellik | LSTM | GRU |
|---------|------|-----|
| Kapı Sayısı | 3 | 2 |
| Eğitim Süresi | Uzun | Kısa |
| Uzun Metinler | Yüksek doğruluk | Yeterli doğruluk |
| Küçük Veri | Ezberleme riski | Daha düşük ezberleme |
| Kaynak Tüketimi | Yüksek | Düşük |

---
## 11. LSTM ile Metin Sınıflandırma Uygulaması

Adım adım bir LSTM metin sınıflandırma modeli oluşturacağız:
1. Veri hazırlama (Tokenization + Padding)
2. Embedding matrisi tanımlama
3. LSTM ağ mimarisi kurma
4. Model derleme ve eğitim
5. Tahmin ve değerlendirme

### 11.1 Veri Hazırlama

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

texts = ['NLP egitimi basladi', 'Yapay zeka modelleri', 'Derin ogrenme sureci']
labels = [1, 1, 0]

# Tokenizer olusturma
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

# Kelime indeks sozlugu
print("Kelime Indeksleri:", tokenizer.word_index)

# Metinleri sayisal dizilere donusturme
sequences = tokenizer.texts_to_sequences(texts)
print("Diziler:", sequences)

# Padding - tum dizileri ayni uzunluga getirme
X = pad_sequences(sequences, maxlen=5)
y = np.array(labels)

print("\nPadded X:")
print(X)
print("\ny:", y)

### 11.2 Kelime Gömme Matrisi Tanımlama

In [ ]:
embedding_dim = 100
vocab_size = len(tokenizer.word_index) + 1

# Embedding matrisi olusturma (gercek projede Word2Vec/GloVe kullanilir)
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    embedding_matrix[i] = np.random.rand(embedding_dim)

print(f"Sozluk boyutu: {vocab_size}")
print(f"Embedding boyutu: {embedding_dim}")
print(f"Embedding matris sekli: {embedding_matrix.shape}")

### 11.3 LSTM Ağ Mimarisinin Kurulması

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()

# Embedding katmani - onceden egitilmis agirliklarla
model.add(Embedding(input_dim=vocab_size,
                    output_dim=embedding_dim,
                    weights=[embedding_matrix],
                    input_length=5,
                    trainable=False))  # Embedding agirliklari donduruluyor

# LSTM katmani - 32 birimlik hafiza
model.add(LSTM(32))

# Cikis katmani - ikili siniflandirma icin sigmoid
model.add(Dense(1, activation='sigmoid'))

model.summary()

### 11.4 Modelin Derlenmesi ve Eğitimi

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(X, y, epochs=10, verbose=1)

### 11.5 Yeni Metin ile Tahmin

In [ ]:
test_text = ['Yapay zeka egitimi']
seq = tokenizer.texts_to_sequences(test_text)
X_test = pad_sequences(seq, maxlen=5)

prediction_prob = model.predict(X_test)[0][0]
print(f'Olumlu Siniflandirma Olasiligi: {prediction_prob:.4f}')
print(f'Tahmin: {"Olumlu" if prediction_prob > 0.5 else "Olumsuz"}')

---
## 12. Model Değerlendirme Metrikleri

### Temel Metrikler

| Metrik | Formül | Kullanım Alanı |
|--------|--------|----------------|
| **Kesinlik (Precision)** | TP / (TP + FP) | Spam filtreleri (yanlış silme olmasın) |
| **Duyarlılık (Recall)** | TP / (TP + FN) | Hastalık teşhisi (vaka kaçırılmasın) |
| **F1 Skoru** | 2 × (P × R) / (P + R) | Dengesiz veri setlerinde en güvenilir ölçüt |

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Model tahminleri
y_pred = model.predict(X)
y_pred_bin = (y_pred > 0.5).astype(int).flatten()

print('Genel Model Dogrulugu:', accuracy_score(y, y_pred_bin))
print()
print('Siniflandirma Raporu:')
print(classification_report(y, y_pred_bin, target_names=['Olumsuz', 'Olumlu']))

In [ ]:
# Hata Matrisi (Confusion Matrix) Gorsellestirme
cm = confusion_matrix(y, y_pred_bin)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Olumsuz', 'Olumlu'],
            yticklabels=['Olumsuz', 'Olumlu'])
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gercek Deger')
plt.title('Hata Matrisi (Confusion Matrix)')
plt.tight_layout()
plt.show()

In [ ]:
# Egitim sureci gorsellestirme
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss grafigi
axes[0].plot(history.history['loss'], 'b-', linewidth=2)
axes[0].set_title('Egitim Kaybi (Loss)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# Accuracy grafigi
axes[1].plot(history.history['accuracy'], 'g-', linewidth=2)
axes[1].set_title('Egitim Dogrulugu (Accuracy)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## NLP Model Seçim Rehberi

| Görev | Önerilen Model | Neden? |
|-------|---------------|--------|
| Basit belge sınıflandırma | TF-IDF + Naive Bayes | Yüksek hız, basit yapı |
| Anlamsal kelime ilişkileri | Word2Vec / GloVe | Kelime benzerlikleri |
| Cümle bütünlüğü analizi | LSTM | Uzun bağımlılıklar |
| Sınırlı donanım | GRU | Daha az parametre |
| Türkçe metin analizi | FastText | Sondan eklemeli dil desteği |
| Genel amaçlı (SOTA) | BERT / Transformer | En yüksek performans |

---
## NLP Uygulamalarındaki Temel Zorluklar

1. **Çok Anlamlılık:** "Yüz" kelimesi → sayı, organ veya yüzmek eylemi
2. **Sözcük Türü Karmaşası:** Bir kelime duruma göre isim veya fiil olabilir
3. **Zamir Referansları:** "O" veya "bu" gibi kelimelerin neyi işaret ettiğini bulmak
4. **Serbest Dizilim:** Kelimelerin yer değiştirmesine rağmen cümlenin aynı anlamı koruması
5. **Dil Düzensizlikleri:** Sosyal medya kullanımı ve yazım hataları dilin standart kurallarını bozar

---

**Notebook Sonu** | BTK Akademi - Yapay Zeka ve Üretken Modeller Atölyesi - Hafta 6